# 02 · Conversations – giving Dummy a memory

A model call is **stateless**. Anything that feels like memory is history that somebody sends again. Foundry offers three ways to handle that:

| Way | You pass | Who keeps the history |
|---|---|---|
| **Manual** | the full message list every time | your code |
| **Response chain** | `previous_response_id` | Foundry (stored responses) |
| **Conversation** | `conversation=<id>` | Foundry (a durable *conversation* object with items) |

Agents (chapter 04) use conversations. Deep dive: [docs/conversations.md](../docs/conversations.md).

In [ ]:
import sys; sys.path.insert(0, '.')
from stark_helpers import *

project_openai = openai_client()      # OpenAI client bound to the Foundry project
CHAT_MODEL = model("chat")
ids_to_delete = {"conversations": [], "responses": []}   # everything this notebook stores; the last cell deletes it

## 1 · No memory
**Goal:** show that two calls know nothing of each other. The first call states a fact, the second asks for it, and nothing links them.

**Expected:** the second answer cannot name the suit: the model says it doesn't know, or asks which suit you mean.

In [ ]:
SUIT_FACT = "My suit is the Mark 7. Remember that."
SUIT_QUESTION = "Which suit did I just mention?"

no_memory_fact = project_openai.responses.create(model=CHAT_MODEL, input=SUIT_FACT)
no_memory_answer = project_openai.responses.create(model=CHAT_MODEL, input=SUIT_QUESTION)
ids_to_delete["responses"] += [no_memory_fact.id, no_memory_answer.id]
print("Q:", SUIT_QUESTION)
print("A:", no_memory_answer.output_text)      # the model has no idea

## 2 · Chain responses with `previous_response_id`
Foundry stores every response (`store=True` is the default). Pointing at the previous one replays its context.

**Goal:** the same two prompts as section 1 (`SUIT_FACT`, `SUIT_QUESTION`), with one change: the second call points at the first.

**Expected:** the answer names the Mark 7.

In [ ]:
chain_start = project_openai.responses.create(model=CHAT_MODEL, input=SUIT_FACT)
chain_answer = project_openai.responses.create(model=CHAT_MODEL, input=SUIT_QUESTION, previous_response_id=chain_start.id)
ids_to_delete["responses"] += [chain_start.id, chain_answer.id]
print("Q:", SUIT_QUESTION)
print("A:", chain_answer.output_text)

### Forking
Because each response is stored, you can branch from an **earlier** one – two futures from the same past. The fork starts from `chain_start` in section 2, so run that first.

**Expected:** the fork answers Mark 42. The main branch (`chain_answer`, not touched) still says Mark 7.

In [ ]:
fork_question = "Actually the suit is the Mark 42. Which suit do I have?"
fork_answer = project_openai.responses.create(model=CHAT_MODEL, input=fork_question, previous_response_id=chain_start.id)
ids_to_delete["responses"].append(fork_answer.id)
print("fork:", fork_answer.output_text)
print("main:", chain_answer.output_text)

## 3 · Conversations
A **conversation** is a durable object holding an ordered list of **items** (messages, tool calls, tool results). Create it once, then pass its id on every call. Foundry appends the new input *and* the model output as items.

**Goal:** a follow-up that only makes sense with the history. The conversation starts with a seed message (who you are, how long answers may be), then two questions follow.

**Expected:** the conversation id, then two answers under 20 words. The second answer names a suit for space, which shows the conversation carried the first question. Re-running the cell starts a new conversation; the clean-up cell at the end deletes them all.

In [ ]:
tony_conversation = project_openai.conversations.create(
    items=[{"type": "message", "role": "user", "content": "I am Tony Stark. Keep answers under 20 words."}],
    metadata={"chapter": "02", "topic": "conversations"},
)
ids_to_delete["conversations"].append(tony_conversation.id)
print("conversation:", tony_conversation.id)

conversation_questions = ["Which suit is the best for underwater missions?", "And which one for space?"]
for question in conversation_questions:
    response = project_openai.responses.create(model=CHAT_MODEL, conversation=tony_conversation.id, input=question)
    ids_to_delete["responses"].append(response.id)
    print(f"\nQ: {question}\nA: {response.output_text}")

### Look inside: the items
`conversations.items.list` returns what Foundry keeps for `tony_conversation` (section 3): every message in and out, with its role.

**Expected:** at least five `message` items: the seed message, then a `user` message and an `assistant` answer for each question. ❓ Not checked: whether the list starts with the newest item, and whether `reasoning` items show up too.

In [ ]:
conversation_items = project_openai.conversations.items.list(tony_conversation.id)
for item in conversation_items:
    text = ""
    if item.type == "message":
        text = " ".join(getattr(part, "text", "") for part in item.content)
    print(f"{item.type:10} {getattr(item, 'role', ''):10} {text}")

### Add items yourself
You can seed context (for example a user profile or a retrieved document) without calling the model. This adds a note to `tony_conversation` (section 3), then asks about it.

**Expected:** avoid green, because Pepper banned it. Re-running the cell adds the note a second time; the answer still says green.

In [ ]:
lab_note = {"type": "message", "role": "user", "content": "Note: Pepper has banned the color green in the lab."}
project_openai.conversations.items.create(tony_conversation.id, items=[lab_note])

color_question = "What color should I avoid in the lab, and why?"
color_answer = project_openai.responses.create(model=CHAT_MODEL, conversation=tony_conversation.id, input=color_question)
ids_to_delete["responses"].append(color_answer.id)
print("Q:", color_question)
print("A:", color_answer.output_text)

## 4 · Don't store: `store=False`
For zero-retention scenarios, ask Foundry not to persist the response. You then carry the context yourself.

**Expected:** a one-word codename, then `not retrievable, as expected:` with the error type (❓ likely `NotFoundError`). Nothing is stored, so nothing goes into `ids_to_delete`.

In [ ]:
codename_prompt = "Give me a one-word codename for a secret mission."
unstored_response = project_openai.responses.create(model=CHAT_MODEL, input=codename_prompt, store=False)
print(unstored_response.output_text)
try:
    project_openai.responses.retrieve(unstored_response.id)
    print("stored (unexpected)")
except Exception as exc:
    print("not retrievable, as expected:", type(exc).__name__)

## 👀 In the portal
Nothing new under **Agents**: conversations are not agents, and without an agent they have no Foundry page. With tracing on (chapter 03) find their traces in **Application Insights**. From chapter 04, an agent's **Traces** tab lists its conversation ids.

> 💡 All pages: [docs/portal-tour.md](../docs/portal-tour.md)

## 🧹 Clean up
Stored responses and conversations stay until you delete them. Retrieve one to prove it exists, then delete everything this notebook created.

In [ ]:
print(project_openai.responses.retrieve(ids_to_delete["responses"][0]).id)
for response_id in ids_to_delete["responses"]:
    try: project_openai.responses.delete(response_id)
    except Exception as exc: print("skip", response_id, type(exc).__name__)
for conversation_id in ids_to_delete["conversations"]:
    project_openai.conversations.delete(conversation_id)
print("cleaned up", len(ids_to_delete["responses"]), "responses and", len(ids_to_delete["conversations"]), "conversations")

## 🎯 Recap
| Situation | Pick |
|---|---|
| Quick follow-up in a script | `previous_response_id` |
| Chat sessions, agents, anything long-lived | `conversation` |
| Branching / what-if | `previous_response_id` from an earlier response |
| No retention allowed | `store=False` and carry history yourself |

> 💡 Next: [03 · Tracing and monitoring](03_tracing_and_monitoring.ipynb) – see these calls in the portal and in Log Analytics.